In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import sys
import time
import csv
import os

# Ensure terminal output supports UTF-8 (important for Windows)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

def scrape_aircraft_specs(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    }

    try:
        response = requests.get(url, headers=headers, timeout=20)
        if response.status_code == 404:
            return None
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    
    tabs = {
        "Info": "specs-general",
        "Size": "specs-dimensions",
        "Weight": "specs-weights",
        "Performance": "specs-performance"
    }

    result = {}

    for tab_name, tab_id in tabs.items():
        tab_data = {}
        section = soup.find(id=tab_id)
        
        if section:
            rows = section.find_all('div', class_=re.compile(r'flex justify-between'))
            
            for row in rows:
                dt = row.find('dt')
                dd = row.find('dd')
                
                if dt and dd:
                    key = dt.get_text(strip=True)
                    
                    x_data = dd.get('x-data', '')
                    if 'JSON.parse' in x_data:
                        try:
                            json_match = re.search(r"JSON\.parse\('(.*?)'\)", x_data)
                            if json_match:
                                json_str = json_match.group(1)
                                json_str = json_str.replace('\\u0022', '"')
                                json_str = json_str.encode('utf-8').decode('unicode_escape')
                                
                                items = json.loads(json_str)
                                value = ' · '.join([item['text'] for item in items if 'text' in item])
                                value = value.replace('\\/', '/')
                            else:
                                value = dd.get_text(strip=True)
                        except:
                            value = dd.get_text(strip=True)
                    else:
                        value = dd.get_text(strip=True)
                    
                    value = ' '.join(value.split())
                    if not value:
                        value = "—"
                    
                    tab_data[key] = value
        
        result[tab_name] = tab_data

    return result

def save_to_csv(all_data, filename="aircraft_specs.csv"):
    if not all_data:
        return

    # Collect all possible field names (Tab + Spec)
    fieldnames = ["Name", "URL"]
    all_keys = set()
    for aircraft in all_data:
        for tab, specs in aircraft['data'].items():
            for spec_key in specs.keys():
                all_keys.add(f"{tab}_{spec_key}")
    
    fieldnames.extend(sorted(list(all_keys)))

    with open(filename, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        
        for aircraft in all_data:
            row = {"Name": aircraft['name'], "URL": aircraft['url']}
            for tab, specs in aircraft['data'].items():
                for spec_key, spec_value in specs.items():
                    row[f"{tab}_{spec_key}"] = spec_value
            writer.writerow(row)
    
    print(f"\nMəlumatlar '{filename}' faylına yazıldı.")

def get_slug(name, all_slugs=None):
    # Some manual overrides for common aircraft
    overrides = {
        "Airbus A300": "airbus-a300-600",
        "Airbus A310": "airbus-a310-300",
        "Airbus A319": "airbus-industrie-a319",
        "Airbus A321": "airbus-industrie-a321",
        "Airbus A330": "airbus-industrie-a330-300",
        "Airbus A340": "airbus-a340-300",
        "Airbus A350 XWB": "airbus-a350-xwb",
        "Airbus A380": "airbus-a380-800",
        "Boeing 737": "boeing-737-800",
        "Boeing 747": "boeing-747-400",
        "Boeing 757": "boeing-757-200",
        "Boeing 767": "boeing-767-300",
        "Boeing 777": "boeing-777-300er",
        "Boeing 787 Dreamliner": "boeing-787-dreamliner",
        "Aérospatiale/BAC Concorde": "aerospatiale-bac-concorde",
        "McDonnell Douglas DC-10": "mcdonnell-douglas-dc-10-30",
        "Boeing 717": "mcdonnell-douglas-md-95-boeing-717",
        "Airbus A220": "airbus-a220-300",
        "Airbus A318": "airbus-a318",
        "McDonnell Douglas MD-11": "mcdonnell-douglas-md-11",
        "Embraer ERJ-145": "embraer-erj-145",
        "Embraer ERJ-170": "embraer-170",
        "Embraer ERJ-190": "embraer-190",
        "COMAC ARJ21": "comac-arj21-700",
        "COMAC C919": "comac-c919",
        "Fokker 50 / 60": "fokker-50",
        "Fokker 70 / 100": "fokker-100",
        "McDonnell Douglas MD-80": "mcdonnell-douglas-md-82",
        "McDonnell Douglas MD-90": "mcdonnell-douglas-md-90-er",
        "De Havilland Canada DHC-8 Dash 8": "de-havilland-canada-dhc-8-402-dash-8",
        "Bristol 175 Britannia": "bristol-type-175-britannia",
        "Convair 990": "convair-990-coronado",
        "BAC 1-11": "bac-one-eleven",
        "BAC Vickers VC-10": "vickers-vc10",
        "Embraer EMB-120 Brasilia": "embraer-emb-120-brasilia",
        "Dornier Do-328": "fairchild-dornier-328jet",
        "ATR 42 / 72": "atr-72-200",
        "Saab 340": "saab-340",
        "Saab 2000": "saab-2000",
        "Lockheed L-1011 TriStar": "lockheed-l-1011-tristar",
        "Lockheed L-188 Electra": "lockheed-l-188-electra",
        "Convair 880": "convair-880",
    }
    
    if name in overrides:
        return overrides[name]
    
    # Generic slugification
    slug = name.lower()
    slug = slug.replace("/", "-").replace(" ", "-").replace(".", "-")
    slug = re.sub(r'-+', '-', slug)
    return slug

if __name__ == "__main__":
    aircraft_list = [
        "Airbus A300", "Airbus A310", "Airbus A319", "Airbus A320", "Airbus A321", 
        "Airbus A330", "Airbus A340", "Airbus A350 XWB", "Airbus A380", 
        "Boeing 737", "Boeing 747", "Boeing 757", "Boeing 767", "Boeing 777", 
        "Boeing 787 Dreamliner", "Aérospatiale/BAC Concorde", "McDonnell Douglas DC-10", 
        "Bristol 175 Britannia", "Convair 880", "Lockheed L-1011 TriStar", 
        "Lockheed L-188 Electra", "De Havilland Canada DHC-8 Dash 8", "Fokker 50 / 60", 
        "British Aerospace BAe 146 / Avro RJ", "De Havilland Canada DHC-7 Dash 7", 
        "McDonnell Douglas DC-9", "McDonnell Douglas MD-80", "McDonnell Douglas MD-90", 
        "Airbus A220", "Bombardier BD-700 Global", "Fokker 70 / 100", 
        "McDonnell Douglas MD-11", "ATR 42 / 72", "Airbus A318", 
        "Bombardier CRJ-100 Series", "Bombardier CRJ-700", "Bombardier CRJ-900", 
        "Bombardier CRJ-1000", "Convair 990", "Dornier Do-328", "Douglas DC-8", 
        "Embraer EMB-120 Brasilia", "Embraer ERJ-145", "Embraer ERJ-170", 
        "Embraer ERJ-190", "Saab 340", "Saab 2000", "VFW-Fokker VFW 614", 
        "BAC 1-11", "BAC Vickers VC-10", "British Aerospace BAe ATP", 
        "British Aerospace Jetstream 31 / 32", "British Aerospace Jetstream 41", 
        "Britten-Norman BN-2 Islander", "De Havilland Canada DHC-6 Twin Otter", 
        "Hawker Siddeley HS-121 Trident", "Vickers Vanguard", "Boeing 720", 
        "Boeing 717", "Short 330", "Embraer E2", "COMAC ARJ21", "COMAC C919", 
        "Handley Page Herald", "Canadair CL-44"
    ]

    results = []
    base_url = "https://aerocorner.com/aircraft/"

    for name in aircraft_list:
        slug = get_slug(name)
        url = f"{base_url}{slug}/"
        
        print(f"Processing: {name} ({url})")
        data = scrape_aircraft_specs(url)
        
        if data:
            results.append({"name": name, "url": url, "data": data})
            print(f"  [+] Success")
        else:
            # Try a search if direct URL fails?
            print(f"  [-] Failed")
        
        # Polite delay
        time.sleep(1)

    if results:
        save_to_csv(results)
    else:
        print("\nHeç bir məlumat tapılmadı.")


Bütün 65 təyyarə üçün məlumatlar toplanır...

Processing: Airbus A300 (https://aerocorner.com/aircraft/airbus-a300-600/)


C:\Users\99455\AppData\Local\Temp\ipykernel_10064\1746533615.py:61: DeprecationWarning: invalid escape sequence '\/'
  json_str = json_str.encode('utf-8').decode('unicode_escape')


  [+] Success
Processing: Airbus A310 (https://aerocorner.com/aircraft/airbus-a310-300/)
  [+] Success
Processing: Airbus A319 (https://aerocorner.com/aircraft/airbus-industrie-a319/)
  [+] Success
Processing: Airbus A320 (https://aerocorner.com/aircraft/airbus-a320/)
  [+] Success
Processing: Airbus A321 (https://aerocorner.com/aircraft/airbus-industrie-a321/)
  [+] Success
Processing: Airbus A330 (https://aerocorner.com/aircraft/airbus-industrie-a330-300/)
  [+] Success
Processing: Airbus A340 (https://aerocorner.com/aircraft/airbus-a340-300/)
  [+] Success
Processing: Airbus A350 XWB (https://aerocorner.com/aircraft/airbus-a350-xwb/)
  [+] Success
Processing: Airbus A380 (https://aerocorner.com/aircraft/airbus-a380-800/)
  [+] Success
Processing: Boeing 737 (https://aerocorner.com/aircraft/boeing-737-800/)
  [+] Success
Processing: Boeing 747 (https://aerocorner.com/aircraft/boeing-747-400/)
  [+] Success
Processing: Boeing 757 (https://aerocorner.com/aircraft/boeing-757-200/)
  [+]

In [4]:
import requests
from bs4 import BeautifulSoup
import json
import re
import sys
import time
import csv

# Windows terminalında UTF-8 dəstəyini təmin etmək üçün
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

def scrape_aircraft_specs(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    }
    try:
        response = requests.get(url, headers=headers, timeout=20)
        if response.status_code == 404: return None
        response.raise_for_status()
    except: return None

    soup = BeautifulSoup(response.text, 'html.parser')
    tabs = {"Info": "specs-general", "Size": "specs-dimensions", "Weight": "specs-weights", "Performance": "specs-performance"}
    result = {}

    for tab_name, tab_id in tabs.items():
        tab_data = {}
        section = soup.find(id=tab_id)
        if section:
            rows = section.find_all('div', class_=re.compile(r'flex justify-between'))
            for row in rows:
                dt, dd = row.find('dt'), row.find('dd')
                if dt and dd:
                    key = dt.get_text(strip=True)
                    x_data = dd.get('x-data', '')
                    if 'JSON.parse' in x_data:
                        try:
                            json_match = re.search(r"JSON\.parse\('(.*?)'\)", x_data)
                            if json_match:
                                json_str = json_match.group(1).replace('\\u0022', '"')
                                json_str = json_str.encode('utf-8').decode('unicode_escape')
                                items = json.loads(json_str)
                                value = ' · '.join([item['text'] for item in items if 'text' in item]).replace('\\/', '/')
                            else: value = dd.get_text(strip=True)
                        except: value = dd.get_text(strip=True)
                    else: value = dd.get_text(strip=True)
                    tab_data[key] = ' '.join(value.split()) if value else "—"
        result[tab_name] = tab_data
    return result

def save_to_csv(all_data, filename="extra_aircraft_specs.csv"):
    if not all_data: return
    fieldnames = ["Name", "URL"]
    all_keys = set()
    for ac in all_data:
        for tab, specs in ac['data'].items():
            for sk in specs.keys(): all_keys.add(f"{tab}_{sk}")
    fieldnames.extend(sorted(list(all_keys)))
    with open(filename, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for ac in all_data:
            row = {"Name": ac['name'], "URL": ac['url']}
            for tab, specs in ac['data'].items():
                for sk, sv in specs.items(): row[f"{tab}_{sk}"] = sv
            writer.writerow(row)
    print(f"\nMəlumatlar '{filename}' faylına yazıldı.")

if __name__ == "__main__":
    # Bu təyyarələr üçün xüsusi URL-lər (Slugs)
    targets = {
        "Bristol 175 Britannia": "bristol-britannia",
        "Lockheed L-1011 TriStar": "lockheed-l-1011-tristar-100",
        "De Havilland Canada DHC-8 Dash 8": "bombardier-q400",
        "British Aerospace BAe 146 / Avro RJ": "british-aero-bae-146",
        "De Havilland Canada DHC-7 Dash 7": "de-havilland-canada-dash-7",
        "McDonnell Douglas DC-9": "mcdonnell-douglas-dc-9-30",
        "Bombardier BD-700 Global": "bombardier-global-express-xrs",
        "ATR 42 / 72": "atr-72-500",
        "Bombardier CRJ-100 Series": "bombardier-crj-100-200",
        "VFW-Fokker VFW 614": "vfw-fokker-614",
        "BAC 1-11": "bac-one-eleven",
        "British Aerospace BAe ATP": "british-aerospace-atp",
        "British Aerospace Jetstream 31 / 32": "british-aerospace-jetstream-31-32",
        "British Aerospace Jetstream 41": "british-aerospace-jetstream-41",
        "Britten-Norman BN-2 Islander": "britten-norman-bn-2-islander",
        "De Havilland Canada DHC-6 Twin Otter": "de-havilland-dhc-6-twin-otter",
        "Boeing 720": "boeing-720-023b",
        "Embraer E2": "embraer-e190-e2",
        "Handley Page Herald": "handley-page-hpr-7-dart-herald"
    }

    results = []
    print(f"Siyahıdakı {len(targets)} təyyarə emal olunur...\n")

    for name, slug in targets.items():
        url = f"https://aerocorner.com/aircraft/{slug}/"
        print(f"Yüklənir: {name}...")
        data = scrape_aircraft_specs(url)
        if data:
            results.append({"name": name, "url": url, "data": data})
            print(f"  [+] Tapıldı.")
        else:
            print(f"  [-] Tapılmadı.")
        time.sleep(1)

    if results: save_to_csv(results)
    else: print("\nMəlumat toplana bilmədi. URL-ləri bir daha yoxlamaq lazımdır.")


Siyahıdakı 19 təyyarə emal olunur...

Yüklənir: Bristol 175 Britannia...


C:\Users\99455\AppData\Local\Temp\ipykernel_10064\1712114712.py:42: DeprecationWarning: invalid escape sequence '\/'
  json_str = json_str.encode('utf-8').decode('unicode_escape')


  [+] Tapıldı.
Yüklənir: Lockheed L-1011 TriStar...
  [+] Tapıldı.
Yüklənir: De Havilland Canada DHC-8 Dash 8...
  [+] Tapıldı.
Yüklənir: British Aerospace BAe 146 / Avro RJ...
  [+] Tapıldı.
Yüklənir: De Havilland Canada DHC-7 Dash 7...
  [-] Tapılmadı.
Yüklənir: McDonnell Douglas DC-9...
  [+] Tapıldı.
Yüklənir: Bombardier BD-700 Global...
  [+] Tapıldı.
Yüklənir: ATR 42 / 72...
  [+] Tapıldı.
Yüklənir: Bombardier CRJ-100 Series...
  [-] Tapılmadı.
Yüklənir: VFW-Fokker VFW 614...
  [-] Tapılmadı.
Yüklənir: BAC 1-11...
  [-] Tapılmadı.
Yüklənir: British Aerospace BAe ATP...
  [+] Tapıldı.
Yüklənir: British Aerospace Jetstream 31 / 32...
  [-] Tapılmadı.
Yüklənir: British Aerospace Jetstream 41...
  [-] Tapılmadı.
Yüklənir: Britten-Norman BN-2 Islander...
  [-] Tapılmadı.
Yüklənir: De Havilland Canada DHC-6 Twin Otter...
  [-] Tapılmadı.
Yüklənir: Boeing 720...
  [-] Tapılmadı.
Yüklənir: Embraer E2...
  [-] Tapılmadı.
Yüklənir: Handley Page Herald...
  [-] Tapılmadı.

Məlumatlar 'extra